# S11 · leakdepth 0.1.1 — validation of the A10 lineage matcher (NB3)

Runs the protocol pre-registered as amendment **A11** (S11_STAGE5_AMENDMENTS.md) before this notebook was built.
leakdepth **0.1.1** makes A10 the default lineage matcher (`matcher="a10"`, `info_margin=20`); the 0.1.0 matcher is
available as `matcher="frozen"`. A10 was introduced after the 0.1.0 negative controls failed (post hoc).

| Cell | Item | What it does |
|---|---|---|
| 2 | E1 (part 1) | dev seeds 0–4: 0.1.1 `frozen` = the 0.1.0 code (taken from the 0.1.0 wheel), and A10 with the bound disabled = `frozen` |
| 3 | D | line-by-line check of `nb2/s2_lineage_heldout.jsonl` (SHA-256, per-cell counts, equality with the NB2 summary) |
| 4–5 | R2 controls + E1 (part 2) | positive control and all 8 negative directions with `frozen` and A10 at 10/15/20/25/30 bits; `frozen` and A10@20 must equal NB2 |
| 6 | R1 + R2 | S2 lineage injection on fresh held-out seeds 200–219, `frozen` and all five margins |
| 7 | R3 | genuine reuse between low-information OpenFlow corpora (planted exact copies), `frozen` and all five margins |
| 8 | Summary | compact JSON with the pre-registered verdicts (paste it back) |

**Runtime.** Colab **High-RAM CPU**; expected 2–3 h in total. Every job is saved to `MyDrive/S11-SecSplit/nb3/` as it
finishes. After a disconnect, rerun cells 1, 2 and 4 (finished parts reload from Drive), then the interrupted cell; finished jobs are skipped. If E1 fails,
cells 6–7 do nothing and the summary reports E1 and D only (pre-registered stop rule).


In [ ]:
# 1. Setup: Drive, embedded leakdepth 0.1.1 wheel and the 0.1.0 wheel (both hash-checked), versions
import base64, hashlib, json, os, subprocess, sys, time, glob, gc, zipfile, io, math, importlib, importlib.util
LOCAL = os.environ.get("NB3_LOCAL")                     # local smoke tests only; unset on Colab
if LOCAL:
    BASE = LOCAL
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
OUT = f"{BASE}/S11-SecSplit/nb3"
NB2 = f"{BASE}/S11-SecSplit/nb2"
RAW6 = f"{BASE}/S06-OFDG/raw"
RAW11 = f"{BASE}/S11-SecSplit/raw"
os.makedirs(OUT, exist_ok=True)
WHEEL_SHA = "13c0954a6ba000c65a44cad280a8f1e59f1fdab63fd7c1048e9a95a9dc617eb6"  # package files identical to the tested wheel 5213ebdf23c2…; author metadata set to the release author
WHEEL_SHA_010 = "6194495b5dff78f7b98794e04870d535404c1c949d4cfa28a54469abd764737d"  # package files identical to the tested wheel 7da15ab81b61…; author metadata set to the release author
WHEEL_B64 = "..."
WHEEL_B64_010 = "..."
WD = LOCAL or "/content"
w = f"{WD}/leakdepth-0.1.1-py3-none-any.whl"
open(w, "wb").write(base64.b64decode(WHEEL_B64))
assert hashlib.sha256(open(w, "rb").read()).hexdigest() == WHEEL_SHA, "0.1.1 wheel hash mismatch"
w0 = base64.b64decode(WHEEL_B64_010)
assert hashlib.sha256(w0).hexdigest() == WHEEL_SHA_010, "0.1.0 wheel hash mismatch"
if not LOCAL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", w, "pyarrow", "huggingface_hub"], check=True)
import numpy as np, pandas as pd, scipy, leakdepth as L
LL = importlib.import_module("leakdepth.lineage")
assert L.__version__ == "0.1.1", L.__version__
# the 0.1.0 lineage module, loaded side by side from the 0.1.0 wheel (only for the E1 dev-seed check)
src010 = zipfile.ZipFile(io.BytesIO(w0)).read("leakdepth/lineage.py").decode()
spec = importlib.util.spec_from_loader("leakdepth.lineage_010", loader=None)
OLD = importlib.util.module_from_spec(spec); OLD.__package__ = "leakdepth"; sys.modules["leakdepth.lineage_010"] = OLD
exec(compile(src010, "leakdepth/lineage_010.py", "exec"), OLD.__dict__)
print("leakdepth", L.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| CPUs", os.cpu_count(), "| RAM GB", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))
T0 = time.time()
MARGINS = [10.0, 15.0, 20.0, 25.0, 30.0]
MODES = ["frozen"] + [f"a10_{int(m)}" for m in MARGINS]
SMOKE = bool(os.environ.get("NB3_SMOKE"))
S2F = f"{OUT}/s2_fresh_200_219.jsonl"
R3F = f"{OUT}/r3_lowinfo.jsonl"
from leakdepth.inject import plant, TRANSFORMS
TFS = list(TRANSFORMS) + ["renameall"]

def kw(mode):
    return {"matcher": "frozen"} if mode == "frozen" else {"matcher": "a10", "info_margin": float(mode.split("_")[1])}

def done(name):
    return os.path.exists(f"{OUT}/{name}")

def jdump(obj, name):
    tmp = f"{OUT}/{name}.tmp"
    json.dump(obj, open(tmp, "w"), default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, f"{OUT}/{name}")

def jload(name, folder=None):
    return json.load(open(f"{folder or OUT}/{name}"))

def counts(rep_dict):
    return [c["matched"] for c in rep_dict["results"]["containment"]]

def detected(rep_dict):
    return any(f["id"] == "lineage-containment" and f["severity"] in ("WARN", "ERROR") for f in rep_dict["findings"])


In [ ]:
# 2. E1 part 1 (dev seeds 0-4): 0.1.1 frozen == 0.1.0 code; A10 with the bound disabled == frozen
if done("E1_dev.json"):
    E1dev = jload("E1_dev.json")
else:
    rows = []
    for s in range(5):
        for tf in (TFS[:3] if SMOKE else TFS):
            for rho in (0.0, 0.05):
                p = plant("lineage", rho, s, transform=tf)
                old = [c["matched"] for c in OLD.lineage(p.train, p.other, "t", "o", seed=0).to_dict()["results"]["containment"]]
                new = counts(L.lineage(p.train, p.other, "t", "o", seed=0, matcher="frozen").to_dict())
                orig = LL.match_rows
                LL.match_rows = lambda *a, **k: orig(*a, **{**k, "info_margin": -1e18})   # bound disabled
                try:
                    off = counts(L.lineage(p.train, p.other, "t", "o", seed=0, matcher="frozen").to_dict())
                finally:
                    LL.match_rows = orig
                rows.append({"seed": s, "transform": tf, "rho": rho, "v010": old, "frozen": new, "a10_off": off})
    E1dev = {"rows": rows, "ok": all(r["v010"] == r["frozen"] == r["a10_off"] for r in rows)}
    jdump(E1dev, "E1_dev.json")
print("E1 dev configs:", len(E1dev["rows"]), "| 0.1.0 == 0.1.1 frozen == A10 bound off:", E1dev["ok"])


In [ ]:
# 3. D: line-by-line check of the NB2 S2 file
S2_EXPECTED_FROZEN = {**{tf: {0.0: 0, 0.01: 13, 0.05: 20, 0.2: 20} for tf in ("reorder", "round3", "float32", "colsubset", "rowsubset", "combo")},
                      "rename": {0.0: 0, 0.01: 12, 0.05: 20, 0.2: 20}, "renameall": {0.0: 0, 0.01: 11, 0.05: 20, 0.2: 20},
                      "noise": {0.0: 0, 0.01: 0, 0.05: 0, 0.2: 0}}
f2 = f"{NB2}/s2_lineage_heldout.jsonl"
raw = open(f2, "rb").read()
lines = [json.loads(x) for x in raw.decode().splitlines() if x.strip()]
keys = {(r["transform"], r["rho"], r["seed"]) for r in lines}
cell = {}
for r in lines:
    c = cell.setdefault(f'{r["transform"]}|{r["rho"]}', [0, 0, 0])
    c[0] += bool(r["frozen_detected"]); c[1] += bool(r["A10_detected"]); c[2] += 1
summ2 = jload("S11_NB2_summary.json", NB2)["S2"]
mism = [k for k, v in cell.items() if summ2.get(k.split("|")[0], {}).get(str(float(k.split("|")[1]))) != v]
mism += [f"{tf}|{rho}" for tf, d in summ2.items() for rho in d if f"{tf}|{float(rho)}" not in cell]
stage5 = [k for k, v in cell.items() if v[0] != S2_EXPECTED_FROZEN[k.split("|")[0]][round(float(k.split("|")[1]), 2)]]
D = {"sha256": hashlib.sha256(raw).hexdigest(), "bytes": len(raw), "lines": len(lines), "unique_keys": len(keys),
     "seeds": [min(k[2] for k in keys), max(k[2] for k in keys)] if keys else None,
     "cells[frozen,A10,n]": cell, "mismatch_vs_NB2_summary": sorted(mism), "frozen_mismatch_vs_stage5": sorted(stage5),
     "frozen_eq_A10_matched_rows": sum(r["frozen_matched"] == r["A10_matched"] for r in lines),
     "A10_matched_gt_planted": sum(1 for r in lines if r["n_leaked"] > 0 and max(r["A10_matched"]) > r["n_leaked"]),
     "frozen_matched_gt_planted": sum(1 for r in lines if r["n_leaked"] > 0 and max(r["frozen_matched"]) > r["n_leaked"])}
D["ok"] = (D["lines"] == 720 and D["unique_keys"] == 720 and not D["mismatch_vs_NB2_summary"] and not D["frozen_mismatch_vs_stage5"])
jdump(D, "D_nb2_s2_check.json")
print({k: v for k, v in D.items() if k != "cells[frozen,A10,n]"})


In [ ]:
# 4. Data: S06 raw copies, IoT-DH v1 and the HF mirrors (as NB2)
FILES6 = {"karmous": "karmous__SDN_DDoS_Dataset.csv", "snt": "snt__SNT_dataset.csv", "ukm": "ukmddosflow__flow_features.csv"}
for f in ["ahuja__dataset_sdn.csv", *FILES6.values()]:
    assert os.path.exists(f"{RAW6}/{f}"), f"missing {RAW6}/{f}"

def read6(nm):
    return pd.read_csv(f"{RAW6}/{FILES6[nm]}", low_memory=False)

def iotdh_v1():
    hits = glob.glob(f"{RAW11}/*IoT-DH*.csv") + glob.glob(f"{RAW11}/*IoT-DH*.zip")
    assert hits, f"IoT-DH v1 not found in {RAW11}"
    p = ([h for h in hits if h.endswith(".csv")] or hits)[0]
    if p.endswith(".zip"):
        z = zipfile.ZipFile(p); name = [n for n in z.namelist() if n.endswith(".csv")][0]
        d = pd.read_csv(z.open(name), low_memory=False)
    else:
        d = pd.read_csv(p, low_memory=False)
    if not LOCAL:
        assert len(d) == 15_817_209, len(d)
    return d

def hf_mirrors():
    if LOCAL:
        return pd.read_parquet(f"{BASE}/cic.parquet"), pd.read_parquet(f"{BASE}/unsw.parquet")
    from huggingface_hub import hf_hub_download
    MANIFEST = [{"repo": "rdpahalavan/CIC-IDS2017", "file": "Network-Flows/CICIDS_Flow.parquet", "sha256": "e64f6adfbcb7f96b018cf36c731be64ef4de3353ccd7340b190590734a0dda35"},
                {"repo": "Mouwiya/UNSW-NB15", "file": "data/train-00000-of-00002.parquet", "sha256": "2aada2a26d061111f4e8fb84e716f5f11264fee71abe04697d42cb89e488d047"},
                {"repo": "Mouwiya/UNSW-NB15", "file": "data/train-00001-of-00002.parquet", "sha256": "7c6699ae967567977dee9e9193543b515255f4e1671ca79bf9ae65e7866ffef1"}]
    P = {}
    for m in MANIFEST:
        p = hf_hub_download(m["repo"], m["file"], repo_type="dataset", local_dir=f"/content/data/{m['repo'].replace('/', '__')}")
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == m["sha256"], "mirror changed"
        P[m["file"]] = p
    return (pd.read_parquet(P["Network-Flows/CICIDS_Flow.parquet"]),
            pd.concat([pd.read_parquet(P[f]) for f in sorted(P) if f.startswith("data/train")], ignore_index=True))

ahuja = pd.read_csv(f"{RAW6}/ahuja__dataset_sdn.csv", low_memory=False)
if not LOCAL:
    assert len(ahuja) == 104_345, len(ahuja)
E1_OK = jload("E1.json")["E1_OK"] if done("E1.json") else None
print("Ahuja rows", len(ahuja))


In [ ]:
# 5. R2 on the real controls (frozen + A10 at five margins) and E1 part 2 (frozen and A10@20 must equal NB2)
CONTROLS = [("negative_ahuja_ukm", "ukm"), ("negative_ahuja_snt", "snt"), ("negative_ahuja_karmous", "karmous"),
            ("negative_cic_unsw", None), ("positive_ahuja_iotdhv1", None)]
NB2_FILES = {"frozen": "lineage_frozen_{}.json", "a10_20": "lineage_A10_{}.json"}

def run_control(name, loader):
    todo = [m for m in MODES if not done(f"ctl_{m}_{name}.json")]
    if todo:
        a, b, na, nb = loader()
        for m in todo:
            t0 = time.time(); d = L.lineage(a, b, na, nb, seed=0, **kw(m)).to_dict(); d["seconds"] = round(time.time() - t0, 1)
            jdump(d, f"ctl_{m}_{name}.json")
        del a, b; gc.collect()
    return {m: jload(f"ctl_{m}_{name}.json") for m in MODES}

CTL, E1ctl = {}, []
for name, nm in CONTROLS:
    if name == "negative_cic_unsw":
        loader = lambda: (*hf_mirrors(), "CIC-IDS2017", "UNSW-NB15")
    elif name.startswith("positive"):
        loader = lambda: (ahuja, iotdh_v1(), "Ahuja", "IoT-DH v1")
    else:
        loader = (lambda nm=nm: (ahuja, read6(nm), "Ahuja", nm))
    R = run_control(name, loader)
    CTL[name] = {m: [(c["direction"], c["matched"], c["sampled_rows"], round(c["containment"], 5), c.get("rejected_low_info"))
                     for c in R[m]["results"]["containment"]] for m in MODES}
    for m, pat in NB2_FILES.items():
        f = f"{NB2}/" + pat.format(name)
        ref = counts(json.load(open(f))) if os.path.exists(f) else None
        E1ctl.append({"control": name, "mode": m, "nb2": ref, "nb3": counts(R[m]), "equal": ref == counts(R[m])})
    print(name, {m: [r[1] for r in v] for m, v in CTL[name].items()})
E1_OK = bool(E1dev["ok"] and all(e["equal"] for e in E1ctl))
jdump({"dev_ok": E1dev["ok"], "controls": E1ctl, "E1_OK": E1_OK}, "E1.json")
print("E1 equivalence gate:", "PASS" if E1_OK else "FAIL -> pre-registered STOP (cells 6-7 skip)")


In [ ]:
# 6. R1 + R2: S2 lineage injection on FRESH held-out seeds 200-219 (frozen + A10 at five margins); resumable
SEEDS = list(range(200, 203)) if SMOKE else list(range(200, 220))
RHOS = (0.0, 0.05) if SMOKE else (0.0, 0.01, 0.05, 0.20)
if not E1_OK:
    print("E1 failed or not run: skipped (pre-registered stop rule)")
else:
    seen = {(r["transform"], r["rho"], r["seed"]) for r in map(json.loads, open(S2F))} if os.path.exists(S2F) else set()
    todo = [(tf, rho, s) for tf in TFS for rho in RHOS for s in SEEDS if (tf, rho, s) not in seen]
    print(len(seen), "done,", len(todo), "to run"); t0 = time.time()
    for i, (tf, rho, s) in enumerate(todo):
        p = plant("lineage", rho, s, transform=tf)
        row = {"transform": tf, "rho": rho, "seed": s, "n_leaked": int(p.n_leaked), "n_other": len(p.other)}
        for m in MODES:
            d = L.lineage(p.train, p.other, "train", "other", seed=0, **kw(m)).to_dict()
            row[m] = {"det": detected(d), "matched": counts(d),
                      "rej": sum((c.get("rejected_low_info") or 0) for c in d["results"]["containment"])}
        with open(S2F, "a") as fh:
            fh.write(json.dumps(row) + "\n")
        if (i + 1) % 50 == 0:
            print(i + 1, f"{time.time() - t0:.0f} s")
    S2 = pd.DataFrame(list(map(json.loads, open(S2F))))
    tab = {m: S2.assign(d=[r["det"] for r in S2[m]]).groupby(["transform", "rho"])["d"].sum().unstack() for m in ("frozen", "a10_20")}
    print(pd.concat(tab, axis=1).to_string())


In [ ]:
# 7. R3: genuine reuse between low-information corpora (exact copies planted), frozen + A10 at five margins
import zlib
N_HOST, N_DONOR = (3000, 30000) if SMOKE else (50_000, 200_000)
R3_RHOS = (0.0, 0.05) if SMOKE else (0.0, 0.01, 0.05, 0.20)
R3_SEEDS = [0] if SMOKE else [0, 1]

def entropy_bits(s):
    p = s.astype(str).value_counts(normalize=True).to_numpy()
    return float(-(p * np.log2(p)).sum())

def pkt_col(df):
    for c in df.columns:
        if str(c).lower().replace("_", "") in ("packetcount", "pktcount"):
            return c
    return None

def label_col(df):
    for c in df.columns:
        if str(c).lower() in ("label", "class", "traffic_classification"):
            return c
    return None

if not E1_OK:
    print("E1 failed or not run: skipped (pre-registered stop rule)")
else:
    karm, snt = read6("karmous"), read6("snt")
    seen = {(r["pair"], r["rho"], r["stratum"], r["colset"], r["transform"], r["seed"]) for r in map(json.loads, open(R3F))} \
        if os.path.exists(R3F) else set()
    meta = {}
    for pair, (dname, donor_full, hname, host_full) in {"snt->karmous": ("snt", snt, "karmous", karm),
                                                         "karmous->snt": ("karmous", karm, "snt", snt)}.items():
        # host columns renamed positionally to the donor's names so planted copies share the donor schema
        k = min(donor_full.shape[1], host_full.shape[1])
        dcols, hcols = list(donor_full.columns[:k]), list(host_full.columns[:k])
        rng0 = np.random.default_rng(12345)
        donor = donor_full.iloc[np.sort(rng0.choice(len(donor_full), min(N_DONOR, len(donor_full)), replace=False))][dcols].reset_index(drop=True)
        host_pool = host_full[hcols].set_axis(dcols, axis=1)
        nonconst = [c for c in dcols if donor[c].nunique(dropna=False) > 1]
        ent = sorted(nonconst, key=lambda c: (entropy_bits(donor[c]), c))
        colsets = {"all": nonconst, "low6": ent[:6], "low4": ent[:4]}
        pc = pkt_col(donor)
        empty_idx = np.flatnonzero(pd.to_numeric(donor[pc], errors="coerce").to_numpy() == 0) if pc is not None else np.array([], int)
        meta[pair] = {"columns_positional_map": dict(zip(hcols, dcols)), "n_columns": k, "colsets": colsets,
                      "entropy_bits": {c: round(entropy_bits(donor[c]), 2) for c in nonconst}, "packet_col": pc,
                      "donor_rows": len(donor), "donor_empty_rows": int(len(empty_idx))}
        for seed in R3_SEEDS:
            rng = np.random.default_rng(1000 + seed)
            host_base = host_pool.iloc[rng.choice(len(host_pool), N_HOST, replace=False)].reset_index(drop=True)
            for rho in R3_RHOS:
                for stratum in ("any", "empty"):
                    pool = np.arange(len(donor)) if stratum == "any" else empty_idx
                    kk = int(round(rho * N_HOST))
                    if kk > len(pool):
                        continue
                    pick = np.sort(rng.choice(pool, kk, replace=False)) if kk else np.array([], int)
                    host = pd.concat([host_base, donor.iloc[pick]], ignore_index=True)
                    planted = np.r_[np.zeros(len(host_base), bool), np.ones(kk, bool)]
                    perm = rng.permutation(len(host)); host = host.iloc[perm].reset_index(drop=True); planted = planted[perm]
                    for cs, cols in colsets.items():
                        dproj, hproj = donor[cols], host[cols]
                        dkey = set(pd.util.hash_pandas_object(dproj.astype(str), index=False).to_numpy().tolist())
                        hkey = pd.util.hash_pandas_object(hproj.astype(str), index=False).to_numpy()
                        chance = int(sum(1 for i, h in enumerate(hkey) if (not planted[i]) and h in dkey))
                        for tf in ("none", "renameall"):
                            key = (pair, rho, stratum, cs, tf, seed)
                            if key in seen:
                                continue
                            hp = hproj
                            if tf == "renameall":
                                krng = np.random.default_rng(zlib.crc32(repr(key).encode()))
                                order = krng.permutation(len(cols))
                                hp = hproj.iloc[:, order].set_axis([f"c{j:02d}" for j in range(len(cols))], axis=1)
                            row = {"pair": pair, "rho": rho, "stratum": stratum, "colset": cs, "transform": tf, "seed": seed,
                                   "n_host": len(host), "planted": kk, "chance_identical_nonplanted": chance}
                            for m in MODES:
                                r = L.lineage_matches(hp, dproj, seed=0, **kw(m))
                                if r["matched"] is None:
                                    row[m] = {"aligned": len(r["aligned"]), "matched": None}
                                    continue
                                mt = np.zeros(len(host), bool); mt[r["index"]] = r["matched"]
                                tp, fp = int((mt & planted).sum()), int((mt & ~planted).sum())
                                row[m] = {"aligned": len(r["aligned"]), "matched": int(mt.sum()), "tp": tp, "fp": fp,
                                          "det": bool(mt.sum() / len(r["index"]) >= 0.01)}
                            with open(R3F, "a") as fh:
                                fh.write(json.dumps(row) + "\n")
            print(pair, "seed", seed, f"{time.time() - T0:.0f} s since start")
    jdump(meta, "r3_meta.json")
    R3 = pd.DataFrame(list(map(json.loads, open(R3F))))
    print(len(R3), "R3 configurations")


In [ ]:
# 8. Compact summary with the pre-registered A11 verdicts (paste this output back); also nb3/S11_NB3_summary.json
summ = {"leakdepth": L.__version__, "E1": jload("E1.json") if done("E1.json") else None,
        "D": {k: v for k, v in jload("D_nb2_s2_check.json").items() if k != "cells[frozen,A10,n]"} if done("D_nb2_s2_check.json") else None}
if summ["D"]:
    summ["D"]["cells[frozen,A10,n]"] = jload("D_nb2_s2_check.json")["cells[frozen,A10,n]"]
if summ["E1"]:
    summ["E1"] = {"E1_OK": summ["E1"]["E1_OK"], "dev_ok": summ["E1"]["dev_ok"],
                  "controls_unequal": [(e["control"], e["mode"], e["nb2"], e["nb3"]) for e in summ["E1"]["controls"] if not e["equal"]]}
verdict = {}
if summ["E1"] and summ["E1"]["E1_OK"]:
    CTL = {name: {m: [(c["direction"], c["matched"], c["sampled_rows"], round(c["containment"], 5), c.get("rejected_low_info"))
                      for c in jload(f"ctl_{m}_{name}.json")["results"]["containment"]] for m in MODES}
           for name in ["negative_ahuja_ukm", "negative_ahuja_snt", "negative_ahuja_karmous", "negative_cic_unsw", "positive_ahuja_iotdhv1"]}
    summ["controls[dir,matched,n,C,rej]"] = CTL
    summ["R3_meta_colsets"] = {k: v["colsets"] for k, v in jload("r3_meta.json").items()} if done("r3_meta.json") else None
    def ctl_ok(m):
        pos = CTL["positive_ahuja_iotdhv1"][m][0][3] >= 0.99
        neg = all(r[3] <= 0.01 for n, v in CTL.items() if n.startswith("negative") for r in v[m])
        return pos, neg
    S2 = pd.DataFrame(list(map(json.loads, open(S2F))))
    s2 = {}
    for m in MODES:
        g = S2.assign(d=[r["det"] for r in S2[m]]).groupby(["transform", "rho"])["d"].agg(["sum", "count"])
        s2[m] = {f"{tf}|{rho}": [int(a), int(b)] for (tf, rho), (a, b) in g.iterrows()}
    summ["S2_fresh[det,n]"] = {m: s2[m] for m in ("frozen", "a10_20")}   # all margins: s2_fresh_200_219.jsonl
    summ["S2_fresh_matched_gt_planted"] = {m: int(sum(1 for _, r in S2.iterrows() if r["n_leaked"] > 0 and max(r[m]["matched"]) > r["n_leaked"])) for m in MODES}
    def s2_ok(m):
        rec = all(v[0] >= 0.95 * v[1] for k, v in s2[m].items() if not k.startswith("noise") and float(k.split("|")[1]) >= 0.05)
        fpr = all(v[0] <= 0.05 * v[1] for k, v in s2[m].items() if float(k.split("|")[1]) == 0.0)
        return rec, fpr
    sweep = {}
    for m in MODES:
        pos, neg = ctl_ok(m); rec, fpr = s2_ok(m)
        sweep[m] = {"positive>=0.99": pos, "negatives<=0.01": neg, "S2_recall>=0.95@rho>=0.05": rec, "S2_FPR<=0.05": fpr,
                    "meets_all": pos and neg and rec and fpr}
    summ["R2_sweep"] = sweep
    verdict["R1_met_a10_20"] = all(s2_ok("a10_20"))
    verdict["R1_rho001_a10_20"] = {k: v for k, v in s2["a10_20"].items() if k.endswith("|0.01")}
    verdict["R2_robust_15_20_25"] = all(sweep[m]["meets_all"] for m in ("a10_15", "a10_20", "a10_25"))
    R3 = pd.DataFrame(list(map(json.loads, open(R3F))))
    agg, lim = {}, []
    for (pair, st, cs, tf, rho), g in R3.groupby(["pair", "stratum", "colset", "transform", "rho"]):
        cellv = {}
        for m in MODES:
            v = [r[m] for r in g.to_dict("records")]
            ok = [x for x in v if x.get("matched") is not None]
            cellv[m] = {"det": sum(x["det"] for x in ok), "n": len(v), "not_aligned": len(v) - len(ok),
                        "recall": round(sum(x["tp"] for x in ok) / max(1, sum(r["planted"] for r, x in zip(g.to_dict("records"), v) if x.get("matched") is not None)), 3),
                        "fp": sum(x["fp"] for x in ok)}
        cellv["chance_identical"] = int(g["chance_identical_nonplanted"].sum())
        agg[f"{pair}|{st}|{cs}|{tf}|{rho}"] = cellv
        if rho >= 0.05 and cs != "all":
            a, f = cellv["a10_20"]["det"] / cellv["a10_20"]["n"], cellv["frozen"]["det"] / cellv["frozen"]["n"]
            if a < 0.5 or f - a >= 0.2:
                lim.append(f"{pair}|{st}|{cs}|{tf}|{rho}: A10 det {a:.2f} vs frozen {f:.2f}")
    jdump(agg, "r3_cells.json")
    summ["R3[pair|stratum|colset|tf|rho]=[det frozen,10,15,20,25,30; n; recall frozen,a10_20; fp frozen,a10_20; chance]"] = {
        k: [*[v[m]["det"] for m in MODES], v["frozen"]["n"], v["frozen"]["recall"], v["a10_20"]["recall"],
            v["frozen"]["fp"], v["a10_20"]["fp"], v["chance_identical"]] for k, v in agg.items() if not k.endswith("|0.01")}
    summ["R3_documented_limitations"] = lim
    verdict["R3_adequate_all_cells"] = all(v["a10_20"]["det"] >= 0.95 * v["a10_20"]["n"] for k, v in agg.items()
                                           if k.split("|")[2] == "all" and float(k.split("|")[4]) >= 0.05)
    verdict["D_ok"] = bool(summ["D"] and summ["D"]["ok"])
    verdict["A10_validated_for_tested_conditions"] = bool(summ["E1"]["E1_OK"] and verdict["D_ok"] and verdict["R1_met_a10_20"]
                                                          and verdict["R2_robust_15_20_25"] and verdict["R3_adequate_all_cells"])
else:
    verdict["STOP"] = "E1 failed or not run: only E1 and D are reported (pre-registered)"
summ["verdict"] = verdict
summ["runtime_min_this_session"] = round((time.time() - T0) / 60, 1)
jdump(summ, "S11_NB3_summary.json")
print(json.dumps(summ, separators=(",", ":"), default=str))
